In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import SVG

from vizopt.animation import SnapshotCallback, snapshots_to_animated_svg
from vizopt.base import OptimConfig
from vizopt.templates.euler.stars_vs_circles import (
    optimize_radially_convex_sets_and_circles_from_graph,
)
from vizopt.examples.sets import make_animals_graph, make_multiples_of_primes_graph
from vizopt.schedules import make_term_schedules

# Sets of multiples

In [ ]:
primes = [2, 3, 5]
G = make_multiples_of_primes_graph(primes)
elements = sorted(n for n in G.nodes if G.out_degree(n) == 0)

In [ ]:
n_iters = 8000
best_params = {
    "collision_delay": 0.27,
    "collision_ramp":  0.33,
    "exclusion_delay": 0.32,
    "exclusion_ramp":  0.47,
    "area_delay":      0.31,
    "area_ramp":       0.27,
    "perimeter_delay": 0.69,
    "perimeter_ramp":  0.45,
    "attraction_peak": 0.69,
    "attraction_ramp": 0.18,
}
term_schedules = make_term_schedules(best_params, n_iters)

In [ ]:
snapshot_cb = SnapshotCallback(every=50)

named_results, named_circles_out, history, problem = optimize_radially_convex_sets_and_circles_from_graph(
    G,
    weight_area=1.0,
    weight_perimeter=1.0,
    weight_exclusion=10.0,
    weight_smoothness=2.0,
    weight_position_anchor=0.3,
    weight_circle_collision=20.0,
    weight_set_attraction=5.0,
    term_schedules=term_schedules,
    optim_config=OptimConfig(n_iters=n_iters, learning_rate=0.002),
    callback=snapshot_cb,
)
print(f"Captured {len(snapshot_cb.snapshots)} snapshots")

In [ ]:
set_colors = ["tomato", "steelblue", "mediumorchid", "mediumseagreen"]
set_node_names = [f"multiples_of_{p}" for p in primes]
set_labels = [f"Multiples of {p}" for p in primes]

fig, ax = plt.subplots(figsize=(9, 9))

for set_name, color, label in zip(set_node_names, set_colors, set_labels):
    result = named_results[set_name]
    cx, cy = result["center"]
    radii = result["radii"]
    angles = result["angles"]
    bx = np.append(cx + radii * np.cos(angles), cx + radii[0] * np.cos(angles[0]))
    by = np.append(cy + radii * np.sin(angles), cy + radii[0] * np.sin(angles[0]))
    ax.fill(bx, by, alpha=0.15, color=color)
    ax.plot(bx, by, color=color, linewidth=2, label=label)
    ax.plot(cx, cy, "+", color=color, markersize=10, markeredgewidth=2)

for n in elements:
    r_orig = G.nodes[n]["r"]
    cx_out, cy_out, _ = named_circles_out[n]
    ax.add_patch(plt.Circle((cx_out, cy_out), r_orig, facecolor="lightyellow", alpha=0.9,
                            edgecolor="dimgray", linewidth=1.5))
    ax.text(cx_out, cy_out, str(n), ha="center", va="center", fontsize=10, fontweight="bold")

ax.set_aspect("equal")
ax.autoscale_view()
ax.margins(0.15)
ax.legend(loc="upper right", fontsize=11)
ax.set_title("Multiples of 2, 3, 5 among 1-11\nStar-shaped boundaries with movable element circles")
ax.axis("off")
plt.tight_layout()

In [ ]:
pd.DataFrame(history).set_index("iteration").plot(marker=".", figsize=(7, 3))
plt.ylabel("Loss value")
plt.title("Optimization history")
plt.tight_layout()

In [ ]:
svg = snapshots_to_animated_svg(problem, snapshot_cb.snapshots, fps=5, size=600, history=history)
SVG(data=svg)

# Animals taxonomy

In [ ]:
G_a = make_animals_graph()
elements_a = sorted(n for n in G_a.nodes if G_a.out_degree(n) == 0)
set_names_a = [n for n in nx.topological_sort(G_a) if G_a.out_degree(n) > 0]

In [ ]:
named_results_a, named_circles_a, history_a, problem_a = (
    optimize_radially_convex_sets_and_circles_from_graph(
        G_a,
        weight_area=1.0,
        weight_perimeter=1.0,
        weight_exclusion=10.0,
        weight_smoothness=2.0,
        weight_position_anchor=0.01,
        weight_circle_collision=100.0,
        weight_set_attraction=5.0,
        term_schedules=term_schedules,
        optim_config=OptimConfig(n_iters=n_iters, learning_rate=0.002),
    )
)

TODO set labeling in the figure

In [ ]:
colors_a = plt.cm.tab10(np.linspace(0, 0.9, len(set_names_a)))

fig, ax = plt.subplots(figsize=(9, 9))

for set_name, color in zip(set_names_a, colors_a):
    result = named_results_a[set_name]
    cx, cy = result["center"]
    radii = result["radii"]
    angles = result["angles"]
    bx = np.append(cx + radii * np.cos(angles), cx + radii[0] * np.cos(angles[0]))
    by = np.append(cy + radii * np.sin(angles), cy + radii[0] * np.sin(angles[0]))
    ax.fill(bx, by, alpha=0.15, color=color)
    ax.plot(bx, by, color=color, linewidth=2, label=set_name)
    ax.plot(cx, cy, "+", color=color, markersize=10, markeredgewidth=2)

for n in elements_a:
    r_orig = G_a.nodes[n]["r"]
    cx_out, cy_out, _ = named_circles_a[n]
    ax.add_patch(plt.Circle((cx_out, cy_out), r_orig, facecolor="lightyellow", alpha=0.9,
                            edgecolor="dimgray", linewidth=1.5))
    ax.text(cx_out, cy_out, n, ha="center", va="center", fontsize=9, fontweight="bold")

ax.set_aspect("equal")
ax.autoscale_view()
ax.margins(0.15)
ax.legend(loc="upper right", fontsize=10)
ax.set_title("Animal taxonomy — star-shaped boundaries with movable element circles")
ax.axis("off")
plt.tight_layout()